In [0]:
# 1. Configure Databricks to unlock your Azure Storage Account
storage_account_name = "adityacapstonestorage" 
"account_key = \"YOUR_AZURE_STORAGE_KEY_HERE\"\n",


# Securely pass the key to the Spark engine
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    account_key
)

print("1. Reading raw CSV files from your Azure 'rawdataset' container...")
raw_path = f"abfss://rawdataset@{storage_account_name}.dfs.core.windows.net/"

# Read the CSVs into PySpark DataFrames
sales_df = spark.read.csv(raw_path + "sales_transactions.csv", header=True, inferSchema=True)
product_df = spark.read.csv(raw_path + "product_master.csv", header=True, inferSchema=True)
store_df = spark.read.csv(raw_path + "store_master.csv", header=True, inferSchema=True)
customer_df = spark.read.csv(raw_path + "customer_data.csv", header=True, inferSchema=True)
inventory_df = spark.read.csv(raw_path + "inventory_data.csv", header=True, inferSchema=True)
clickstream_df = spark.read.csv(raw_path + "clickstream_events.csv", header=True, inferSchema=True)

print("2. Saving Managed Bronze Delta Tables directly to 'medallion.bronze'...")

# We now use catalog.schema.table syntax to write directly to your new setup!
# Capstone Requirement: Partition sales by order_date and channel
sales_df.write.format("delta").mode("overwrite").partitionBy("order_date", "channel").saveAsTable("medallion.bronze.sales_transactions")

# Save the rest of the tables
product_df.write.format("delta").mode("overwrite").saveAsTable("medallion.bronze.product_master")
store_df.write.format("delta").mode("overwrite").saveAsTable("medallion.bronze.store_master")
customer_df.write.format("delta").mode("overwrite").saveAsTable("medallion.bronze.customer_data")
inventory_df.write.format("delta").mode("overwrite").saveAsTable("medallion.bronze.inventory_data")
clickstream_df.write.format("delta").mode("overwrite").saveAsTable("medallion.bronze.clickstream_events")

print("Success! Raw data read from Azure and safely loaded into medallion.bronze.")

1. Reading raw CSV files from your Azure 'rawdataset' container...
2. Saving Managed Bronze Delta Tables directly to 'medallion.bronze'...
Success! Raw data read from Azure and safely loaded into medallion.bronze.


In [0]:
# ─── INGESTION METADATA LOG ───────────────────────────────────────────────────
from datetime import datetime
from pyspark.sql import Row

print("Logging ingestion metadata...")

# Count rows from each table we just loaded
log_rows = [
    Row(run_id="bronze_run_001", table_name="sales_transactions",  rows_loaded=sales_df.count(),      source_path=raw_path + "sales_transactions.csv",  loaded_at=str(datetime.now()), status="success"),
    Row(run_id="bronze_run_001", table_name="product_master",      rows_loaded=product_df.count(),    source_path=raw_path + "product_master.csv",       loaded_at=str(datetime.now()), status="success"),
    Row(run_id="bronze_run_001", table_name="store_master",        rows_loaded=store_df.count(),      source_path=raw_path + "store_master.csv",         loaded_at=str(datetime.now()), status="success"),
    Row(run_id="bronze_run_001", table_name="customer_data",       rows_loaded=customer_df.count(),   source_path=raw_path + "customer_data.csv",        loaded_at=str(datetime.now()), status="success"),
    Row(run_id="bronze_run_001", table_name="inventory_data",      rows_loaded=inventory_df.count(),  source_path=raw_path + "inventory_data.csv",       loaded_at=str(datetime.now()), status="success"),
    Row(run_id="bronze_run_001", table_name="clickstream_events",  rows_loaded=clickstream_df.count(),source_path=raw_path + "clickstream_events.csv",   loaded_at=str(datetime.now()), status="success"),
]

log_df = spark.createDataFrame(log_rows)

log_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("medallion.bronze.ingestion_log")

print("✅ Ingestion log saved to medallion.bronze.ingestion_log")
log_df.show(truncate=False)

Logging ingestion metadata...
✅ Ingestion log saved to medallion.bronze.ingestion_log
+--------------+------------------+-----------+------------------------------------------------------------------------------------+--------------------------+-------+
|run_id        |table_name        |rows_loaded|source_path                                                                         |loaded_at                 |status |
+--------------+------------------+-----------+------------------------------------------------------------------------------------+--------------------------+-------+
|bronze_run_001|sales_transactions|1000500    |abfss://rawdataset@adityacapstonestorage.dfs.core.windows.net/sales_transactions.csv|2026-04-03 06:02:31.290370|success|
|bronze_run_001|product_master    |500        |abfss://rawdataset@adityacapstonestorage.dfs.core.windows.net/product_master.csv    |2026-04-03 06:02:31.730737|success|
|bronze_run_001|store_master      |100        |abfss://rawdataset@adityaca

In [0]:
%sql
-- Compact the small files created by partitioning + speed up joins
OPTIMIZE medallion.bronze.sales_transactions
  ZORDER BY (product_id, customer_id);

OPTIMIZE medallion.bronze.clickstream_events
  ZORDER BY (customer_id, product_id);

-- Turn on auto-optimize for future loads so you don't have to run this manually again
ALTER TABLE medallion.bronze.sales_transactions
  SET TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
  );

ALTER TABLE medallion.bronze.clickstream_events
  SET TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
  );

SELECT 'Bronze OPTIMIZE complete' AS status;

status
Bronze OPTIMIZE complete


In [0]:
%sql
SELECT * FROM medallion.bronze.sales_transactions LIMIT 20;

transaction_id,order_date,channel,store_id,product_id,customer_id,quantity,unit_price,discount,total_amount,payment_type,ingestion_timestamp
5009060,2024-03-23,Store,30,1027,107714,1,704.2830668255222,0.17,584.55,Credit Card,2024-03-23
5009203,2024-03-23,Store,97,1111,102012,5,465.00439611687773,0.26,1720.52,Credit Card,2024-03-24
5010177,2024-03-23,Store,61,1007,65803,5,333.3901572687763,0.07,1550.26,Debit Card,2024-03-23
5012081,2024-03-23,Store,59,1046,105395,2,913.6890476934449,0.2,1461.9,Cash,2024-03-24
5013143,2024-03-23,Store,10,1178,21693,3,795.8292432385986,0.18,1957.74,Debit Card,2024-03-23
5014197,2024-03-23,Store,1,1440,87408,5,670.7895269203686,0.1,3018.55,Cash,2024-03-23
5015148,2024-03-23,Store,50,1337,81480,1,427.7614644102844,0.24,325.1,Debit Card,2024-03-25
5015627,2024-03-23,Store,49,1020,16118,3,298.78688969106366,0.01,887.4,UPI,2024-03-24
5015959,2024-03-23,Store,14,1145,108639,3,312.6800440269077,0.2,750.43,Credit Card,2024-03-24
5016004,2024-03-23,Store,47,1293,88900,5,752.1298427699535,0.2,3008.52,Cash,2024-03-23


In [0]:
%sql
SELECT * FROM medallion.bronze.customer_data 
WHERE age IS NULL;

customer_id,gender,age,loyalty_status,signup_channel,city
10294,Male,null,None,Online,Bangalore
10337,Male,null,None,Store,Chandigarh
11084,Male,null,Silver,Mobile,Bangalore
11532,Female,null,Gold,Online,Pune
11846,Male,null,None,Online,Bangalore
12285,Male,null,None,Online,Mumbai
12466,Male,null,Gold,Online,Hyderabad
12787,Female,null,Silver,Online,Chandigarh
13779,Female,null,None,Online,Chandigarh
14180,Male,null,Gold,Store,Chandigarh


In [0]:
%sql
SELECT * FROM medallion.bronze.sales_transactions 
WHERE quantity < 0 OR unit_price = 0;

transaction_id,order_date,channel,store_id,product_id,customer_id,quantity,unit_price,discount,total_amount,payment_type,ingestion_timestamp
5259834,2024-11-01,Mobile,6,1411,88552,-1,277.1204117645859,0.03,1075.23,Credit Card,2024-11-02
5843459,2024-10-14,Store,99,1077,54558,-1,772.8468301875885,0.22,1808.46,UPI,2024-10-14
5010075,2024-06-19,Online,31,1084,79876,1,0.0,0.27,513.04,Cash,2024-06-21
5594969,2024-11-16,Online,28,1010,36959,6,0.0,0.19,1231.54,UPI,2024-11-16
5773439,2024-01-13,Mobile,84,1084,108149,2,0.0,0.15,1194.76,Credit Card,2024-01-14
5958457,2024-03-16,Mobile,58,1431,100894,-1,865.0502411841719,0.03,4195.49,Credit Card,2024-03-17
5773712,2024-04-25,Store,66,1156,97599,-1,461.62726789198643,0.11,410.85,Credit Card,2024-04-25
5726921,2024-02-07,Mobile,72,1055,64296,-1,440.3005993865963,0.27,1285.68,Cash,2024-02-07
5437365,2024-12-02,Online,56,1384,63937,5,0.0,0.01,1558.43,UPI,2024-12-02
5165993,2024-12-23,Mobile,25,1049,20003,6,0.0,0.27,1060.54,Credit Card,2024-12-24


In [0]:
%sql
SELECT transaction_id, COUNT(*) as duplicate_count 
FROM medallion.bronze.sales_transactions 
GROUP BY transaction_id 
HAVING duplicate_count > 1;

transaction_id,duplicate_count
5039129,2
5903108,2
5820166,2
5539977,2
5965055,2
5850598,2
5299286,2
5796809,2
5550860,2
5548404,2


In [0]:
%sql
SELECT * FROM medallion.bronze.inventory_data 
WHERE stock_on_hand < 0;

product_id,store_id,stock_on_hand,reorder_level,last_updated
1018,74,-5,35,2026-03-29T07:03:39.796371Z
1020,61,-5,62,2026-03-29T07:03:39.796438Z
1030,35,-5,40,2026-03-29T07:03:39.796769Z
1067,32,-5,85,2026-03-29T07:03:39.797858Z
1071,40,-5,57,2026-03-29T07:03:39.79797Z
1094,81,-5,90,2026-03-29T07:03:39.798639Z
1094,37,-5,23,2026-03-29T07:03:39.798644Z
1103,81,-5,47,2026-03-29T07:03:39.798925Z
1107,77,-5,94,2026-03-29T07:03:39.799025Z
1109,88,-5,100,2026-03-29T07:03:39.799089Z


In [0]:
%sql
SELECT DISTINCT city, state 
FROM medallion.bronze.store_master 
ORDER BY city;

city,state
Bangalore,Telangana
Bangalore,Punjab
Bangalore,Maharashtra
Bangalore,Delhi
Bangalore,Karnataka
Bangalore,Rajasthan
Chandigarh,Delhi
Chandigarh,Punjab
Chandigarh,Karnataka
Chandigarh,Telangana


In [0]:
%sql
SELECT DISTINCT brand, category, sub_category 
FROM medallion.bronze.product_master 
ORDER BY brand;

brand,category,sub_category
Adidas,Beauty,Furniture
Adidas,Electronics,Furniture
Adidas,Beauty,Laptop
Adidas,Beauty,Cosmetics
Adidas,Electronics,Cosmetics
Adidas,Beauty,Fitness
Adidas,Home,Furniture
Adidas,Electronics,Fitness
Adidas,Sports,Fitness
Adidas,Electronics,Mobile
